# Equipment Usage Queries

## Setup

### Import Libraries

In [ ]:
print("Initializing Python script...\n")

print("Importing libraries...\n")
# Standard Library
import os
from pathlib import Path
import sys

#3rd Party
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

### Settings
#### Set SQL Environment, Import Config File, Define Notebook Settings

In [ ]:
# # Define environment
# os.environ['ENV'] = 'production'

# # # Set system path
# # sys.path.append(r'\\ut02sa7\ac\~Cody\~Projects\python\monorepo\shared\global_config')

# # # Import config file variables
# # import config

# # Toggle max rows and max columns
# # pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

In [ ]:
# print(config.ENV)
# print(config.ACTIVE_SERVER)
# print(config.ENVIRONMENTS)

### Connect to Pyrometry Database on DEV/QUAL/PROD Server

In [ ]:
# env_server = config.PROD_SERVER
# env_database = config.PYRO_DATABASE
# pyro_server = config.PYRO_SERVER
env_server = 'UT02SQLPRODAGL'
env_database = 'Pyrometry'
pyro_server = 'c30-ywvdb81200.northgrum.com'

print(f'Connecting to {env_server} server {env_database} database...\n')

info_conn_str = (
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    f'Server={env_server};'
    f'Database={env_database};'
    'Trusted_Connection=yes;'
)

info_conn_str = f'mssql+pyodbc://{env_server}/{env_database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server'

info_engine = create_engine(info_conn_str)

equip_pc_query = 'SELECT * FROM [Equipment PCs]'
equip_pcs = pd.read_sql(equip_pc_query, info_engine)
equip_pcs = equip_pcs[equip_pcs['PC'].str.contains('-A')]

## Extract Data and Build Tables

### SQL Server Database Connection & Extraction

#### Build runs_data & parts_data Dataframes

In [ ]:
print("Connecting to Pyrometry Server databases...\n")

conn_str = f'mssql+pyodbc://{pyro_server}/master?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server'
engine = create_engine(conn_str)

query_get_databases = """
    SELECT name
    FROM sys.databases
    WHERE name LIKE 'CPC_%'
        AND name NOT LIKE 'CPC_Staging'
"""

runs_df_list = []
realtime_df_list = []
parts_df_list = []

try:
    with engine.connect() as connection:
        databases = [row[0] for row in connection.execute(text(query_get_databases))]

    for db_name in databases:
        db_conn_str = f'mssql+pyodbc://{pyro_server}/{db_name}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server'
        db_engine = create_engine(db_conn_str)

        with db_engine.connect() as db_connection:
            try:
                # Check for column existence
                press_exists = db_connection.execute(text("""
                    SELECT COUNT(*) 
                    FROM INFORMATION_SCHEMA.COLUMNS 
                    WHERE TABLE_NAME = 'RealtimeData' AND COLUMN_NAME = 'Press';
                """)).fetchone()[0] > 0
                
                purge_exists = db_connection.execute(text("""
                    SELECT COUNT(*) 
                    FROM INFORMATION_SCHEMA.COLUMNS 
                    WHERE TABLE_NAME = 'RealtimeData' AND COLUMN_NAME = 'PurgeStatus';
                """)).fetchone()[0] > 0

                # Fetch Runs
                runs_df = pd.read_sql(f'SELECT * FROM {db_name}.dbo.Runs ORDER BY StartTime', db_connection)
                if not runs_df.empty:
                    runs_df_list.append(runs_df)

                # Construct and Execute Real-time Data Query
                purge_condition = ", SUM(CASE WHEN PurgeStatus = -1 THEN 1 ELSE 0 END) AS PurgeTime" if purge_exists else ""
                where_condition = "WHERE RunStatus = -1" + (" OR PurgeStatus = -1" if purge_exists else "")
                
                query_realtime_data = f"""
                    SELECT Datafile,
                        MIN(CASE WHEN SEG >= 2 THEN Program END) AS Program,
                        MIN(Recipe) AS Recipe,
                        MAX(ROUND(RunTime, 1)) AS RunTime,
                        MAX(ROUND(TSet_Air, 1)) AS Max_Air,
                        MAX(ROUND(TSet_Part, 1)) AS Max_Part,
                        CAST(MAX(SEG) AS INT) AS Max_Seg
                        {purge_condition}
                    FROM {db_name}.dbo.RealtimeData
                    {where_condition}
                    GROUP BY Datafile
                """

                realtime_df = pd.read_sql(query_realtime_data, db_connection)
                if not realtime_df.empty:
                    realtime_df_list.append(realtime_df)

                # Fetch Parts Data
                parts_df = pd.read_sql(f"""
                    SELECT 
                        r.RunID,
                        r.Equipment, 
                        r.Datafile, 
                        r.StartTime,
                        p.PartIndex, 
                        p.ParentPartIndex, 
                        p.PartNumber, 
                        p.PartName, 
                        p.SerialNumber
                    FROM {db_name}.dbo.Runs r
                    INNER JOIN {db_name}.dbo.Parts p ON r.RunID = p.RunID
                """, db_connection)
                if not parts_df.empty:
                    parts_df_list.append(parts_df)

            except Exception as e:
                print(f'Error processing database {db_name}: {e}')

except Exception as e:
    print(f'Error connecting to the database or retrieving databases: {e}')


if runs_df_list:
    combined_runs_df = pd.concat(runs_df_list, ignore_index=True)
    combined_runs_df = combined_runs_df.drop_duplicates(subset=['Datafile'], keep='first')

if realtime_df_list:
    combined_realtime_df = pd.concat(realtime_df_list, ignore_index=True)

if parts_df_list:
    combined_parts_df = pd.concat(parts_df_list, ignore_index=True)

#### Clean parts_data Dataframe

In [ ]:
combined_parts_df = pd.merge(combined_parts_df, combined_realtime_df, on='Datafile')

combined_parts_df['Datafile'] = combined_parts_df['Datafile'].str.replace('.DAT', '', regex=False)
combined_parts_df['Recipe'] = combined_parts_df['Recipe'].str.split('\\').str[-1]
combined_parts_df['Date'] = combined_parts_df['StartTime'].dt.date

combined_part_count_df = combined_parts_df.groupby(['RunID', 'Datafile']).agg(
    total_parts=('PartIndex', 'count'),
    non_zero_unique_parents=('ParentPartIndex', lambda x: x[x != 0].nunique())
).reset_index()

combined_part_count_df['PartCount'] = combined_part_count_df['total_parts'] - combined_part_count_df['non_zero_unique_parents']
combined_part_count_df = combined_part_count_df.drop_duplicates(subset=['Datafile'], keep='first')
parts_data = combined_parts_df[['Equipment', 'Date', 'Datafile', 'Program', 'Recipe', 'PartNumber', 'PartName', 'SerialNumber']]
# parts_data

#### Clean runs_data Dataframe

In [ ]:
# Merge with Equipment PCs & RealtimeData tables
combined_df_ = pd.merge(combined_runs_df, equip_pcs, left_on='Equipment', right_on='Equip', how='left')
combined_df_2 = pd.merge(combined_df_, combined_realtime_df, on='Datafile')

# Additional processing
combined_df_2['Date'] = combined_df_2['StartTime'].dt.date
combined_df_2['PurgeTime'] = combined_df_2['PurgeTime'].fillna(0)
combined_df_2['Recipe'] = combined_df_2['Recipe_x']
combined_df_2['Equipment'] = combined_df_2['Alt_Name']
combined_df_2['Datafile'] = combined_df_2['Datafile'].str.replace('.DAT', '', regex=False)
combined_df_2['Duration'] = combined_df_2['Duration'].round(1)
combined_df_2['Avg_Duration'] = combined_df_2.groupby(['Equipment', 'Recipe', 'RecipeRevision'])['Duration'].transform('median').round(1)
combined_df_2['Recipe'] = combined_df_2['Recipe'].str.split('\\').str[-1]
combined_df_2['PurgeEndTime'] = combined_df_2['EndTime'] + pd.to_timedelta(combined_df_2['PurgeTime'], unit='m')

# Calculate time differences
combined_df_2['next_StartTime'] = combined_df_2['StartTime'].shift(-1)
combined_df_2['CureToCure'] = ((combined_df_2['next_StartTime'] - combined_df_2['EndTime']).dt.total_seconds() / 60).round(1)
combined_df_2['DoorToDoor'] = ((combined_df_2['next_StartTime'] - combined_df_2['PurgeEndTime']).dt.total_seconds() / 60).round(1)

# Determine if runs are aborted
combined_df_2['Aborted'] = np.where(
    (combined_df_2['Duration'] < (combined_df_2['Avg_Duration'] * 0.5)) & (combined_df_2['Duration'] > 5),
    'Yes', 'No'
)

combined_df_2 = pd.merge(combined_df_2, combined_part_count_df, on='Datafile', how='left')
combined_df_2['RunID'] = combined_df_2['RunID_x']

final_columns = [
    'RunID', 'Datafile', 'Equipment', 'Site', 'Recipe', 'RecipeRevision', 'Program', 
    'Date', 'StartTime', 'EndTime', 'Duration', 'Avg_Duration', 'Operator', 
    'Max_Air', 'Max_Part', 'Max_Seg', 'PurgeTime', 'PurgeEndTime', 'Aborted', 
    'DoorToDoor', 'CureToCure', 'PartCount'
]
combined_df_2 = combined_df_2[final_columns]

# Remove duplicates based on Datafile
runs_data = combined_df_2.drop_duplicates(subset=['Datafile'], keep='first')
runs_data

#### Build current_equip_status Dataframe

In [ ]:
query_get_databases = """
    SELECT name 
    FROM sys.databases 
    WHERE name LIKE 'CPC_%' 
    AND name NOT LIKE 'CPC_Staging'
"""

# Function to check column existence
def check_column_exists(db_connection, column_name):
    check_query = f"""
        SELECT COUNT(*) 
        FROM INFORMATION_SCHEMA.COLUMNS 
        WHERE TABLE_NAME = 'RealtimeData' 
        AND COLUMN_NAME = '{column_name}';
    """
    result = db_connection.execute(text(check_query))
    return result.fetchone()[0] > 0

df_list = []  # List to store DataFrames

try:
    with engine.connect() as connection:
        result = connection.execute(text(query_get_databases))
        databases = [row[0] for row in result]

    for db_name in databases:
        db_conn_str = f'mssql+pyodbc://{pyro_server}/{db_name}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL Server'
        db_engine = create_engine(db_conn_str)

        with db_engine.connect() as db_connection:
            try:
                # Check for column existence
                press_exists = check_column_exists(db_connection, 'Press')
                purge_exists = check_column_exists(db_connection, 'PurgeStatus')

                # Construct the main query dynamically
                query_realtime_data = f"""
                    SELECT TOP (1)
                        DataTime AS Time,
                        Datafile,
                        Program,
                        Recipe,
                        ROUND(RunTime, 1) AS RunTime,
                        ROUND(AirTC, 1) AS AirTC,
                        ROUND(TSet_Air, 1) AS AirSet,
                        {"ROUND(Press, 1) AS Press," if press_exists else "0 AS Press,"}
                        {"ROUND(PSet, 1) AS PressSet," if press_exists else "0 AS PressSet,"}
                        ROUND(HITC, 1) AS HiTC,
                        ROUND(LoTC, 1) AS LoTC,
                        ROUND(TSet_Part, 1) AS PartSet,
                        CASE
                            WHEN RunStatus = -1 THEN 'Running'
                            WHEN AbortStatus = -1 THEN 'Aborting'
                            {"WHEN PurgeStatus = -1 THEN 'Purging' " if purge_exists else ''} 
                            ELSE 'Idle'
                        END AS RunStatus,
                        SEG AS Seg,
                        ROUND(SegTime, 1) AS SegTime,
                        ROUND(SegTimeLeft, 1) AS SegTimeLeft
                    FROM {db_name}.dbo.RealtimeData
                    ORDER BY DataTime DESC
                """
                
                # Execute the main query
                result = db_connection.execute(text(query_realtime_data))
                df = pd.DataFrame(result.fetchall(), columns=result.keys())
                if not df.empty:
                    df_list.append(df)

            except Exception as e:
                            print(f'Error processing database {db_name}: {e}')

except Exception as e:
    print(f'Error connecting to the database or retrieving databases: {e}')

# Combine results if any
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    combined_df['Datafile'] = combined_df['Datafile'].str.replace('.DAT', '', regex=False)
    combined_df['Recipe'] = combined_df['Recipe'].str.split('\\').str[-1]

#### Clean current_equip_status Dataframe

In [ ]:
combined_df['Equipment'] = combined_df['Datafile'].str.split(r'[-_]').str[0]
combined_df = combined_df.sort_values(by='Time', ascending=False).reset_index(drop=True)
combined_df_ = pd.merge(combined_df, equip_pcs, left_on='Equipment', right_on='DAT_Name', how='left')
combined_df_['Equipment'] = combined_df_['Alt_Name']

combined_df_ = combined_df_[['Time', 'Equipment', 'Site', 'Datafile', 'Program', 'Recipe', 'RunTime', 'RunStatus', 'AirTC', 'AirSet', 'Press', 'PressSet', 'HiTC', 'LoTC', 'PartSet']]
current_equip_status = combined_df_.sort_values(by=['Site', 'Equipment']).reset_index(drop=True)


latest_runs_data = runs_data.loc[runs_data.groupby(['Equipment', 'Recipe'])['StartTime'].idxmax()]
current_equip_status = pd.merge(
    current_equip_status, 
    latest_runs_data[['Equipment', 'Recipe', 'Avg_Duration']], 
    on=['Equipment', 'Recipe'], 
    how='left')

current_equip_status['TimeLeft'] = (current_equip_status['Avg_Duration'] - current_equip_status['RunTime']).round(0).fillna(-99999).astype(int)
current_equip_status['TimeLeft'] = current_equip_status['TimeLeft'].replace(-99999, 'FIRST RUN')
current_equip_status['TimeLeft'] = current_equip_status['TimeLeft'].apply(lambda x: f'{x:,}' if isinstance(x, int) and x != 'FIRST RUN' else x)
# current_equip_status['Avg_Duration'] = current_equip_status['Avg_Duration'].round(1)
current_equip_status['Avg_Duration'] = current_equip_status['Avg_Duration'].fillna('FIRST RUN')

current_equip_status

#### Build uptime_data Dataframe

In [ ]:
results = []
minutes_in_day = 1440
uptime_adjustment = 216

uptime_analysis = runs_data[['Datafile', 'Equipment', 'Site', 'Recipe', 'Program', 'StartTime', 
                                 'EndTime', 'Duration', 'PurgeTime', 'PurgeEndTime']].copy()

uptime_analysis['Duration+Purge'] = uptime_analysis['Duration'] + uptime_analysis['PurgeTime']

# Process each row in uptime_analysis
for index, row in uptime_analysis.iterrows():
    site = row['Site']
    equipment = row['Equipment']
    start_time = row['StartTime']
    end_time = row['EndTime']
    
    # Calculate RealUptime for the first day
    if start_time.date() == end_time.date():
        RealUptime = row['Duration']
        RealDowntime = minutes_in_day - RealUptime
        results.append({
            'Site': site,
            'Equipment': equipment,
            'Date': start_time.date(),
            'RealUptime': RealUptime,
            'RealDowntime': RealDowntime
        })
    else:
        end_of_day = start_time.replace(hour=23, minute=59, second=59)
        minutes_first_day = (end_of_day - start_time).total_seconds() / 60
        minutes_second_day = (end_time - end_time.replace(hour=0, minute=0)).total_seconds() / 60
        total_RealUptime = minutes_first_day + minutes_second_day
        full_days = (end_time.date() - start_time.date()).days - 1
        total_RealUptime += full_days * minutes_in_day
        
        RealDowntime_first_day = minutes_in_day - minutes_first_day
        RealDowntime_second_day = minutes_in_day - minutes_second_day
        
        results.append({
            'Site': site,
            'Equipment': equipment,
            'Date': start_time.date(),
            'RealUptime': minutes_first_day,
            'RealDowntime': RealDowntime_first_day
        })
        
        results.append({
            'Site': site,
            'Equipment': equipment,
            'Date': end_time.date(),
            'RealUptime': minutes_second_day,
            'RealDowntime': RealDowntime_second_day
        })
        
        for day in range(1, full_days + 1):
            full_day_date = (start_time + pd.Timedelta(days=day)).date()
            results.append({
                'Site': site,
                'Equipment': equipment,
                'Date': full_day_date,
                'RealUptime': minutes_in_day,
                'RealDowntime': 0
            })

# Handle gaps of days between cures
for i in range(len(uptime_analysis) - 1):
    current_end_time = uptime_analysis.iloc[i]['EndTime']
    next_start_time = uptime_analysis.iloc[i + 1]['StartTime']
    
    if (next_start_time - current_end_time).days > 1:
        gap_days = (next_start_time.date() - current_end_time.date()).days - 1
        for day in range(gap_days):
            gap_day_date = (current_end_time + pd.Timedelta(days=day + 1)).date()
            results.append({
                'Site': site,
                'Equipment': uptime_analysis.iloc[i]['Equipment'],
                'Date': gap_day_date,
                'RealUptime': 0,
                'RealDowntime': minutes_in_day
            })

results_df = pd.DataFrame(results)

# Group by date and sum RealUptime and RealDowntime
summary_df = results_df.groupby(['Site', 'Equipment', 'Date']).agg({'RealUptime': 'sum'}).reset_index()
summary_df['RealDowntime'] = minutes_in_day - summary_df['RealUptime']

# Calculate AdjustedUptime and AdjustedDowntime
summary_df['AdjustedUptime'] = summary_df['RealUptime'] + uptime_adjustment
summary_df['AdjustedUptime'] = summary_df['AdjustedUptime'].clip(upper=minutes_in_day)
summary_df['AdjustedDowntime'] = minutes_in_day - summary_df['AdjustedUptime']

# Ensure all columns are integers
summary_df['RealUptime'] = summary_df['RealUptime'].astype(int)
summary_df['RealDowntime'] = summary_df['RealDowntime'].astype(int)
summary_df['AdjustedUptime'] = summary_df['AdjustedUptime'].astype(int)
summary_df['AdjustedDowntime'] = summary_df['AdjustedDowntime'].astype(int)

summary_df['Date'] = pd.to_datetime(summary_df['Date'])

uptime_data = summary_df

#### Build sorting_table Dataframe

In [ ]:
def extract_parts(alt_name):
    alpha_part = ''.join(filter(str.isalpha, alt_name))
    num_part = ''.join(filter(str.isdigit, alt_name))
    return alpha_part, int(num_part) if num_part else np.nan

sorting_table = current_equip_status.copy()
sorting_table[['AlphaPart', 'NumPart']] = sorting_table['Equipment'].apply(extract_parts).apply(pd.Series)
sorting_table_sorted = sorting_table.sort_values(by=['AlphaPart', 'NumPart']).reset_index(drop=True)

sorting_table_sorted['Sorting#'] = range(1, len(sorting_table_sorted) + 1)
sorting_table = sorting_table.merge(sorting_table_sorted[['Equipment', 'Sorting#']], on='Equipment', how='left')
sorting_table = sorting_table.drop(columns=['AlphaPart', 'NumPart'])
sorting_table = sorting_table.sort_values(by='Sorting#')
sorting_table = sorting_table[['Equipment', 'Site', 'Sorting#']].reset_index(drop=True)
sorting_table

## Output to Database

In [ ]:
current_equip_status.to_sql('EquipUsage_Live', con=info_engine, if_exists='replace', index=False)
parts_data.to_sql('EquipUsage_Parts', con=info_engine, if_exists='replace', index=False)
runs_data.to_sql('EquipUsage_Runs', con=info_engine, if_exists='replace', index=False)
sorting_table.to_sql('EquipUsage_Sorting', con=info_engine, if_exists='replace', index=False)
uptime_data.to_sql('EquipUsage_Uptime', con=info_engine, if_exists='replace', index=False)